In [ ]:
import pandas as pd
import random

# Templates for generating job application emails
templates = [
    "Dear Hiring Manager, I am writing to express my interest in the {role} position advertised on {platform}. My resume is attached for your review. Sincerely, {name}",
    "Hello, I am applying for the {role} role I saw on {platform}. Please find my cover letter and CV attached. Thank you, {name}",
    "To whom it may concern, this is my application for the {role} position. I believe my skills are an excellent match for your team. My qualifications are detailed in my attached resume. Regards, {name}",
    "I would like to formally apply for the {role} opening. My experience in this field makes me a strong candidate. Please see my attached CV. Best, {name}"
]

roles = ["Software Engineer", "Data Analyst", "Project Manager", "UX Designer", "Product Owner"]
platforms = ["LinkedIn", "your company website", "Indeed", "Glassdoor"]
names = ["Alex Johnson", "Maria Garcia", "Chen Wei", "Samira Khan"]

job_applications = []
for _ in range(200): # Generate 200 examples
    template = random.choice(templates)
    role = random.choice(roles)
    platform = random.choice(platforms)
    name = random.choice(names)
    text = template.format(role=role, platform=platform, name=name)
    job_applications.append({'label': 'job_application', 'text': text})

# Create a DataFrame
df_synthetic = pd.DataFrame(job_applications)

print("Generated Synthetic Data Sample:")
print(df_synthetic.head())

Generated Synthetic Data Sample:
             label                                               text
0  job_application  Dear Hiring Manager, I am writing to express m...
1  job_application  To whom it may concern, this is my application...
2  job_application  Hello, I am applying for the UX Designer role ...
3  job_application  Hello, I am applying for the Software Engineer...
4  job_application  To whom it may concern, this is my application...


In [ ]:
import pandas as pd
import random
import pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

print("--- Starting Model Training ---")

# --- 1. Load Base Dataset ---
try:
    df_real = pd.read_csv('spam.csv', encoding='latin-1')
    # Clean up the real dataset
    df_real = df_real[['v1', 'v2']]
    df_real.columns = ['label', 'text']
    # Map 'ham' to 'personal' for better categorization
    df_real['label'] = df_real['label'].map({'ham': 'personal', 'spam': 'spam'})
    print(f"Loaded base dataset. Shape: {df_real.shape}")
    print("Base dataset labels:\n", df_real['label'].value_counts())

except FileNotFoundError:
    print("Error: 'spam.csv' not found. Please download it from Kaggle and place it in the 'backend' directory.")
    exit()

# --- 2. Generate Synthetic 'Job Application' Data ---
templates = [
    "Dear Hiring Manager, I am writing to express my interest in the {role} position advertised on {platform}. My resume is attached for your review. Sincerely, {name}",
    "Hello, I am applying for the {role} role I saw on {platform}. Please find my cover letter and CV attached. Thank you, {name}",
    "To whom it may concern, this is my application for the {role} position. I believe my skills are an excellent match for your team. My qualifications are detailed in my attached resume. Regards, {name}",
    "I would like to formally apply for the {role} opening. My experience in this field makes me a strong candidate. Please see my attached CV. Best, {name}"
]
roles = ["Software Engineer", "Data Analyst", "Project Manager", "UX Designer", "Product Owner"]
platforms = ["LinkedIn", "your company website", "Indeed", "Glassdoor"]
names = ["Alex Johnson", "Maria Garcia", "Chen Wei", "Samira Khan"]

job_applications = []
for _ in range(300): # Generate 300 examples for a stronger signal
    template = random.choice(templates)
    text = template.format(
        role=random.choice(roles),
        platform=random.choice(platforms),
        name=random.choice(names)
    )
    job_applications.append({'label': 'job_application', 'text': text})

df_synthetic = pd.DataFrame(job_applications)
print(f"\nGenerated 300 synthetic 'job_application' emails.")


# --- 3. Combine Datasets ---
df_combined = pd.concat([df_real, df_synthetic], ignore_index=True)
print(f"\nCombined dataset shape: {df_combined.shape}")
print("Final dataset labels:\n", df_combined['label'].value_counts())


# --- 4. Train the Model ---
print("\n--- Training the Classification Model ---")

# Define features (X) and target (y)
X = df_combined['text']
y = df_combined['label']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Create a model pipeline: TF-IDF -> Logistic Regression
# TF-IDF converts text into numerical features.
# Logistic Regression is a robust and fast classification algorithm.
model = make_pipeline(TfidfVectorizer(stop_words='english'), LogisticRegression(max_iter=1000))

# Train the model
model.fit(X_train, y_train)
print("Model training complete.")


# --- 5. Evaluate the Model ---
print("\n--- Evaluating Model Performance ---")
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))


# --- 6. Save the Model ---
model_filename = 'your_model.pkl'
with open(model_filename, 'wb') as f:
    pickle.dump(model, f)

print(f"\n✅ Model successfully trained and saved as '{model_filename}'")
print("You can now run the FastAPI backend, which will automatically load this model.")

--- Starting Model Training ---
Error: 'spam.csv' not found. Please download it from Kaggle and place it in the 'backend' directory.

Generated 300 synthetic 'job_application' emails.


NameError: name 'df_real' is not defined

In [3]:
# ==============================================================================
#  Personal Email Classifier Training Notebook
# ==============================================================================
# This notebook will:
# 1. Download the Enron email dataset for foundational training.
# 2. Create a custom dataset for our "Job Application" category.
# 3. Combine the datasets and preprocess the text.
# 4. Train a TF-IDF + Logistic Regression model.
# 5. Evaluate the model's performance.
# 6. Save the final, trained model to a 'your_model.pkl' file.
# ==============================================================================

# Part 1: Setup and Imports
import os
import tarfile
import requests
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

# ==============================================================================
# Part 2: Download and Load the Enron Dataset
# ==============================================================================
# We'll use a pre-processed version of the Enron dataset for convenience.
DATASET_URL = "https://www.cs.cmu.edu/~enron/enron_mail_20150507.tar.gz"
DATASET_PATH = "enron_mail_20150507.tar.gz"
EXTRACTED_PATH = "maildir"

def download_enron_dataset():
    """Downloads the Enron dataset if not already present."""
    if not os.path.exists(DATASET_PATH):
        print(f"Downloading dataset from {DATASET_URL}...")
        response = requests.get(DATASET_URL, stream=True)
        with open(DATASET_PATH, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print("Download complete.")
    else:
        print("Dataset already downloaded.")

    if not os.path.exists(EXTRACTED_PATH):
        print("Extracting dataset...")
        with tarfile.open(DATASET_PATH, "r:gz") as tar:
            tar.extractall()
        print("Extraction complete.")
    else:
        print("Dataset already extracted.")

def load_enron_emails(path=EXTRACTED_PATH, sample_size=2000):
    """Loads a sample of emails from the extracted Enron directory."""
    emails = []
    # We'll just take emails from a few users to create a 'work' category
    # You could expand this to create more categories (e.g., 'personal')
    # by parsing folder names.
    users = ['dasovich-j', 'germany-c', 'shackleton-s', 'skilling-j']
    for user in users:
        user_path = os.path.join(path, user)
        for root, _, files in os.walk(user_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)
                try:
                    with open(file_path, 'r', encoding='latin-1') as f:
                        content = f.read()
                        # Simple preprocessing to get the email body
                        body = content.split('\n\n', 1)[1] if '\n\n' in content else content
                        emails.append({'text': body, 'label': 'work'})
                except Exception as e:
                    pass # Ignore files that can't be read

    # Take a random sample to keep training fast
    df = pd.DataFrame(emails)
    return df.sample(n=min(len(df), sample_size), random_state=42)

download_enron_dataset()
enron_df = load_enron_emails()
print(f"Loaded {len(enron_df)} emails from the Enron dataset.")
print("Enron Data Sample:")
print(enron_df.head())


# ==============================================================================
# Part 3: Create Custom "Job Application" Data
# ==============================================================================
# This is where we teach the model our new category.
# Add more examples to make it even better!
custom_data = [
    {'text': "Dear Hiring Manager, I am writing to express my interest in the Software Engineer position advertised on LinkedIn.", 'label': 'job_application'},
    {'text': "Thank you for your application for the Product Manager role. We are currently reviewing all applications and will be in touch shortly.", 'label': 'job_application'},
    {'text': "Following up on my application for the Data Scientist role. I am very enthusiastic about the opportunity to join your team.", 'label': 'job_application'},
    {'text': "This email is to confirm that we have received your application for the position of UX Designer.", 'label': 'job_application'},
    {'text': "We would like to invite you to an interview for the Marketing Associate position on Tuesday at 3 PM. Please let us know if this time works for you.", 'label': 'job_application'},
    {'text': "After careful consideration, we have decided not to move forward with your candidacy at this time. We wish you the best in your job search.", 'label': 'job_application'},
    {'text': "My resume is attached for your review. I am confident that my skills and experience are a great match for this role.", 'label': 'job_application'},
    {'text': "Thank you for your time during the interview process. I am excited about the possibility of working at your company.", 'label': 'job_application'},
    {'text': "This is an automated response to your job application. We appreciate your interest in our company.", 'label': 'job_application'},
    {'text': "Your application has been moved to the next stage. Our recruiter will reach out to you to schedule a phone screen.", 'label': 'job_application'},
    # Add a few "other" non-job related emails to help the model differentiate.
    {'text': "Hey, are we still on for lunch tomorrow? Let me know.", 'label': 'personal'},
    {'text': "Your Amazon order of a new book has shipped!", 'label': 'personal'},
    {'text': "Hi team, please find the attached quarterly report. Let's discuss it in our meeting on Friday.", 'label': 'work'},
]

custom_df = pd.DataFrame(custom_data)
print(f"\nCreated {len(custom_df)} custom email samples.")
print("Custom Data Sample:")
print(custom_df.head())


# ==============================================================================
# Part 4: Combine and Prepare the Final Dataset
# ==============================================================================
final_df = pd.concat([enron_df, custom_df], ignore_index=True)

# Clean up any missing values
final_df.dropna(inplace=True)

# Shuffle the dataset to ensure the model doesn't learn any order
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nTotal combined dataset size: {len(final_df)}")
print("Label distribution:")
print(final_df['label'].value_counts())

# Split the data into features (X) and labels (y)
X = final_df['text']
y = final_df['label']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"\nTraining set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")


# ==============================================================================
# Part 5: Build and Train the Model Pipeline
# ==============================================================================
# We create a pipeline to chain the vectorizer and the classifier together.
# This makes the model easier to manage and deploy.
model_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_df=0.9, min_df=3)),
    ('clf', LogisticRegression(random_state=42, C=1.0, solver='liblinear'))
])

print("\nTraining the model...")
model_pipeline.fit(X_train, y_train)
print("Training complete!")


# ==============================================================================
# Part 6: Evaluate the Model
# ==============================================================================
print("\nEvaluating model performance on the test set...")
y_pred = model_pipeline.predict(X_test)

# Print the classification report
# This shows precision, recall, and f1-score for each category.
# Look for a good f1-score for 'job_application'!
print(classification_report(y_test, y_pred))


# ==============================================================================
# Part 7: Save the Trained Model
# ==============================================================================
# This is the final step. The saved file is what you will use in your FastAPI backend.
MODEL_FILENAME = "your_model.pkl"
with open(MODEL_FILENAME, 'wb') as f:
    pickle.dump(model_pipeline, f)

print(f"\n✅ Model successfully trained and saved to '{MODEL_FILENAME}'!")
print("You can now download this file from the Colab file explorer (left sidebar) and place it in your backend directory.")

KeyboardInterrupt: 

In [7]:
import pandas as pd
import numpy as np

# Load the dataset with the correct encoding
df = pd.read_csv('spam.csv', encoding='latin1')

print("Dataset loaded successfully!")
df.head()

Dataset loaded successfully!


,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [8]:
# The dataset has a weird 'Email No.' column, let's look at the real columns
# The actual text is in a column that is inconveniently named 'text'
df.rename(columns={'text': 'email_text', 'spam': 'label'}, inplace=True)
df = df[['email_text', 'label']]

# Relabel: 0 will be 'personal', 1 will be 'spam'
df['label'] = df['label'].map({0: 'personal', 1: 'spam'})

print("Original dataset label counts:")
print(df['label'].value_counts())
df.head()

KeyError: "None of [Index(['email_text', 'label'], dtype='object')] are in the [columns]"

In [10]:
# ==============================================================================
import pandas as pd
import re

# Download and unzip the dataset
!kaggle datasets download -d uciml/sms-spam-collection-dataset
!unzip sms-spam-collection-dataset.zip

# Load the dataset using 'latin1' encoding to avoid errors
df = pd.read_csv('spam.csv', encoding='latin1')

# Clean up the dataframe
df = df[['v1', 'v2']] # Select only the relevant columns
df.columns = ['label', 'email_text'] # Rename columns for clarity

# Map labels to more descriptive names
df['label'] = df['label'].map({'ham': 'personal', 'spam': 'spam'})

print("Dataset loaded and pre-processed:")
print(df['label'].value_counts())
print("-" * 30)
print(df.head())


# ==============================================================================
# Step 3: Create Synthetic "Job Application" Data
# ==============================================================================
synthetic_job_apps = [
    {"email_text": "Dear Hiring Manager, I am writing to apply for the Software Engineer position advertised on your careers page. My resume is attached for your review. Sincerely, John Doe", "label": "job_application"},
    {"email_text": "Subject: Application for Marketing Manager. Hello, please find my cover letter and CV attached. I am very interested in this job opening. Thank you, Jane Smith", "label": "job_application"},
    {"email_text": "To whom it may concern, this is my application for the Data Analyst role. I have extensive experience in SQL and Python. My portfolio is available upon request. Best, Alex Ray", "label": "job_application"},
    {"email_text": "I would like to formally apply for the Graphic Designer position. My resume is attached. I look forward to hearing from you soon.", "label": "job_application"},
    {"email_text": "Subject: Job Application: Project Manager. Dear Team, I am a certified PMP with 5 years of experience and am excited about this opportunity. Attaching my resume for your consideration. Regards, Emily White", "label": "job_application"},
    {"email_text": "Following up on my application for the Web Developer role. Please let me know if you require any further information. Thanks, Mike Johnson", "label": "job_application"},
    {"email_text": "Dear Recruiter, I am excited to apply for the UI/UX Designer position I saw on your company website. My portfolio is attached for your review. Thank you for your time and consideration.", "label": "job_application"}
]

df_jobs = pd.DataFrame(synthetic_job_apps)

# Combine original and synthetic data
df_combined = pd.concat([df, df_jobs], ignore_index=True)

# Shuffle the dataset to ensure model doesn't learn from order
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nCombined dataset with synthetic data:")
print(df_combined['label'].value_counts())
print("-" * 30)

# ==============================================================================
# Step 4: Text Cleaning Function
# ==============================================================================
def clean_text(text):
    text = text.lower() # Lowercase text
    text = re.sub(r'http\S+', '', text) # Remove URLs
    text = re.sub(r'[^a-z\s]', '', text) # Remove punctuation and numbers
    text = re.sub(r'\s+', ' ', text).strip() # Remove extra whitespace
    return text

df_combined['email_text'] = df_combined['email_text'].apply(clean_text)

# ==============================================================================
# Step 5: Train and Evaluate the Model
# ==============================================================================
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

X = df_combined['email_text']
y = df_combined['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Create a model pipeline using Logistic Regression
model = make_pipeline(TfidfVectorizer(stop_words='english'), LogisticRegression(max_iter=1000, random_state=42))

print("Training the model...")
model.fit(X_train, y_train)
print("Training complete!")

# Evaluate the model
y_pred = model.predict(X_test)
print("\nModel Performance Evaluation:\n")
print(classification_report(y_test, y_pred))

# ==============================================================================
# Step 6: Save the Model and Download
# ==============================================================================
# ==============================================================================
# Step 6: Save the Model and Download
# ==============================================================================
import joblib
from google.colab import files  # <-- ADD THIS IMPORT

# Save the trained model pipeline to a file
model_filename = 'email_classifier_model.joblib'
joblib.dump(model, model_filename)

print(f"\nModel saved as '{model_filename}'")

# Now this line will work correctly
files.download(model_filename)

Traceback (most recent call last):
  File "/usr/local/bin/kaggle", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/cli.py", line 68, in main
    out = args.func(**command_args)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 1741, in dataset_download_cli
    with self.build_kaggle_client() as kaggle:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 688, in build_kaggle_client
    username=self.config_values['username'],
             ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'username'
unzip:  cannot find or open sms-spam-collection-dataset.zip, sms-spam-collection-dataset.zip.zip or sms-spam-collection-dataset.zip.ZIP.
Dataset loaded and pre-processed:
label
personal    4825
spam         747
Name: count, dtype: int64
------------------------------
      label          

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>